In [ ]:
# -*- coding: utf-8 -*-
"""
============================================================
🛡️ SCORE CV AI - PIPELINE END-TO-END (Code Final Complet)
============================================================

Architecture :
CV PDF/DOCX -> Text Extraction -> Data Cleaning / NLP -> NER ->
Relationship Extraction -> Feature Engineering -> TF-IDF ->
Cosine Similarity -> JSearch API -> Matching Model -> Score CV AI -> Dashboard

Installation :
pip install streamlit pdfplumber python-docx scikit-learn requests

Lancement :
streamlit run app.py
"""

import re
from typing import Dict, Any, List
import streamlit as st

# ============================================================
# IMPORTS OPTIONNELS
# ============================================================

try:
    import pdfplumber
except ImportError:
    pdfplumber = None

try:
    from docx import Document
except ImportError:
    Document = None

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False


# ============================================================
# 1. CONFIGURATION STREAMLIT
# ============================================================

st.set_page_config(
    page_title="Score CV AI Analyse intelligente des CV et matching automatisé avec les offres d’emploi",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# 2. CSS PERSONNALISÉ
# ============================================================

st.markdown(
    """
    <style>
    .stApp {
        background-color: #f5f7f9;
    }
    .main .block-container {
        max-width: 1400px;
        padding-top: 1.5rem;
        padding-bottom: 2rem;
    }
    .header-banner {
        background: linear-gradient(135deg, #064e3b 0%, #047857 55%, #10b981 100%);
        padding: 26px 30px;
        border-radius: 16px;
        margin-bottom: 24px;
        color: white;
        box-shadow: 0 8px 25px rgba(6, 78, 59, 0.15);
    }
    .header-title {
        font-size: 28px;
        font-weight: 800;
        margin: 0;
    }
    .header-subtitle {
        color: #d1fae5;
        font-size: 13px;
        margin-top: 8px;
        line-height: 1.5;
    }
    .pipeline-step {
        background: white;
        border: 1px solid #e2e8f0;
        border-radius: 8px;
        padding: 10px 8px;
        font-size: 10px;
        font-weight: 700;
        color: #047857;
        text-align: center;
        min-height: 45px;
        display: flex;
        align-items: center;
        justify-content: center;
    }
    .score-container {
        background: linear-gradient(135deg, #ecfdf5 0%, #ffffff 100%);
        border: 1px solid #a7f3d0;
        border-radius: 14px;
        padding: 22px;
        box-shadow: 0 4px 15px rgba(16, 185, 129, 0.08);
    }
    .score-number {
        color: #047857;
        font-size: 46px;
        font-weight: 800;
        line-height: 1;
        margin: 8px 0;
    }
    .score-tag {
        display: inline-block;
        background: #d1fae5;
        color: #065f46;
        padding: 5px 11px;
        border-radius: 12px;
        font-size: 11px;
        font-weight: 700;
    }
    .pill-badge {
        display: inline-block;
        padding: 5px 10px;
        border-radius: 12px;
        background-color: #ecfdf5;
        color: #047857;
        font-size: 12px;
        font-weight: 600;
        margin: 3px;
        border: 1px solid #d1fae5;
    }
    .pill-badge-missing {
        display: inline-block;
        padding: 5px 10px;
        border-radius: 12px;
        background-color: #fef2f2;
        color: #dc2626;
        font-size: 12px;
        font-weight: 600;
        margin: 3px;
        border: 1px solid #fecaca;
    }
    .stButton > button {
        width: 100%;
        border-radius: 10px;
        border: none;
        background: linear-gradient(135deg, #047857 0%, #10b981 100%);
        color: white;
        font-weight: 700;
        padding: 11px 20px;
        transition: 0.2s;
    }
    .stButton > button:hover {
        background: linear-gradient(135deg, #065f46 0%, #059669 100%);
        box-shadow: 0 4px 12px rgba(4, 120, 87, 0.25);
    }
    [data-testid="stMetric"] {
        background: white;
        border: 1px solid #e2e8f0;
        padding: 12px;
        border-radius: 10px;
    }
    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# 3. TAXONOMIES
# ============================================================

SKILL_TAXONOMY = [
    "Python", "Docker", "Kubernetes", "Git", "Data Science", "NLP", "Spark",
    "Cloud", "SQL", "Machine Learning", "Deep Learning", "TensorFlow", "PyTorch",
    "Scikit-learn", "Pandas", "NumPy", "Power BI", "Excel", "AWS", "Azure",
    "GCP", "Java", "React"
]

SOFT_SKILLS_TAXONOMY = [
    "leadership", "communication", "teamwork", "organisation", "autonomie",
    "adaptabilité", "créativité", "résolution de problèmes", "travail d'équipe",
    "gestion d'équipe"
]

LANGUAGE_MAP = {
    "français": "Français",
    "francais": "Français",
    "anglais": "Anglais",
    "english": "Anglais",
    "espagnol": "Espagnol",
    "allemand": "Allemand"
}


# ============================================================
# 4. ÉTAPE 1-2 : TEXT EXTRACTION
# ============================================================

def step_1_2_extract_text(uploaded_file) -> str:
    if uploaded_file is None:
        return ""

    file_name = uploaded_file.name.lower()
    text = ""

    try:
        if file_name.endswith(".pdf"):
            if pdfplumber is None:
                st.error("pdfplumber est requis pour lire les PDF.")
                return ""
            with pdfplumber.open(uploaded_file) as pdf:
                for page in pdf.pages:
                    extracted = page.extract_text()
                    if extracted:
                        text += extracted + "\n"

        elif file_name.endswith(".docx"):
            if Document is None:
                st.error("python-docx est requis pour lire les fichiers DOCX.")
                return ""
            document = Document(uploaded_file)
            text = "\n".join(
                p.text for p in document.paragraphs if p.text.strip()
            )

    except Exception as error:
        st.error(f"Erreur pendant l'extraction du CV : {error}")
        return ""

    return text.strip()


# ============================================================
# 5. ÉTAPE 3 : DATA CLEANING / NLP
# ============================================================

def step_3_nlp_cleaning(raw_text: str) -> str:
    if not raw_text:
        return ""

    text = raw_text.lower()

    # Suppression URLs / HTML
    text = re.sub(r"http\S+|www\S+|<.*?>", " ", text)

    # Conservation des caractères techniques (C++, C#, .NET)
    text = re.sub(r"[^\w\s\+\#\.-]", " ", text)

    # Normalisation espaces
    text = re.sub(r"\s+", " ", text).strip()

    stopwords = {
        "le", "la", "les", "un", "une", "des", "du", "de", "et", "en", "pour",
        "avec", "dans", "sur", "par", "est", "sont", "au", "aux", "the", "and",
        "or", "in", "at", "for", "with"
    }

    words = [word for word in text.split() if word not in stopwords]
    return " ".join(words)


# ============================================================
# 6. ÉTAPE 4 : NER / ENTITY EXTRACTION
# ============================================================

def step_4_ner_extraction(raw_text: str, cleaned_text: str) -> Dict[str, Any]:
    entities = {
        "candidate_name": "Candidat Détecté",
        "email": "Non spécifié",
        "phone": "Non spécifié",
        "years_experience": 0,
        "skills": [],
        "soft_skills": [],
        "languages": []
    }

    if not raw_text:
        return entities

    # EMAIL
    email_match = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", raw_text)
    if email_match:
        entities["email"] = email_match.group(0)

    # PHONE
    phone_match = re.search(r"(?:\+?\d[\d\s().-]{7,}\d)", raw_text)
    if phone_match:
        entities["phone"] = phone_match.group(0).strip()

    # EXPERIENCE
    exp_matches = re.findall(
        r"(\d+)\s*(?:ans?|years?)\s*(?:d['’]?expérience|of experience)?",
        raw_text,
        re.IGNORECASE
    )
    if exp_matches:
        valid_exp = [int(v) for v in exp_matches if int(v) < 40]
        if valid_exp:
            entities["years_experience"] = max(valid_exp)

    # NOM CANDIDAT (Heuristique sur les premières lignes)
    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
    for line in lines[:10]:
        if (
            len(line.split()) in [2, 3]
            and not any(char.isdigit() for char in line)
            and "@" not in line
        ):
            entities["candidate_name"] = line
            break

    # HARD SKILLS
    for skill in SKILL_TAXONOMY:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, cleaned_text):
            entities["skills"].append(skill)

    # SOFT SKILLS
    for soft in SOFT_SKILLS_TAXONOMY:
        pattern = r"\b" + re.escape(soft) + r"\b"
        if re.search(pattern, cleaned_text):
            entities["soft_skills"].append(soft.capitalize())

    # LANGUES
    for keyword, language in LANGUAGE_MAP.items():
        pattern = r"\b" + re.escape(keyword) + r"\b"
        if re.search(pattern, cleaned_text):
            if language not in entities["languages"]:
                entities["languages"].append(language)

    return entities


# ============================================================
# 7. ÉTAPE 5 : RELATIONSHIP EXTRACTION
# ============================================================

def step_5_relationship_extraction(
    cv_entities: Dict[str, Any],
    job_cleaned_text: str
) -> Dict[str, Any]:

    relationships = {
        "required_skills_in_job": [],
        "matched_relationships": [],
        "missing_relationships": [],
        "experience_alignment": "Alignement Neutre"
    }

    for skill in SKILL_TAXONOMY:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, job_cleaned_text):
            relationships["required_skills_in_job"].append(skill)

            if skill in cv_entities["skills"]:
                relationships["matched_relationships"].append({
                    "skill": skill,
                    "status": "CONFIRMED",
                    "weight": 1.0
                })
            else:
                relationships["missing_relationships"].append({
                    "skill": skill,
                    "status": "GAP",
                    "weight": 0.0
                })

    # Alignement expérience
    required_experience = re.findall(r"(\d+)\s*(?:ans?|years?)", job_cleaned_text, re.IGNORECASE)
    if required_experience:
        required = max(int(v) for v in required_experience)
        candidate_exp = cv_entities["years_experience"]

        if candidate_exp >= required:
            relationships["experience_alignment"] = "Expérience suffisante"
        elif candidate_exp > 0:
            relationships["experience_alignment"] = "Expérience partiellement alignée"
        else:
            relationships["experience_alignment"] = "Expérience non détectée"

    return relationships


# ============================================================
# 8. ÉTAPE 6 : FEATURE ENGINEERING
# ============================================================

def step_6_feature_engineering(
    cv_entities: Dict[str, Any],
    relationships: Dict[str, Any],
    cv_clean: str,
    job_clean: str
) -> Dict[str, float]:

    total_required = len(relationships["required_skills_in_job"])
    matched_required = len(relationships["matched_relationships"])

    skill_match_ratio = (matched_required / total_required) if total_required > 0 else 0.0
    soft_skills_score = min(len(cv_entities["soft_skills"]) / 3.0, 1.0)
    text_length_ratio = min(len(cv_clean) / max(len(job_clean), 1), 2.0)

    return {
        "skill_match_ratio": skill_match_ratio,
        "soft_skills_score": soft_skills_score,
        "text_length_ratio": text_length_ratio,
        "total_required_skills": float(total_required),
        "matched_required_skills": float(matched_required)
    }


# ============================================================
# 9. ÉTAPE 7-8 : TF-IDF & COSINE SIMILARITY
# ============================================================

def step_7_8_tfidf_cosine(cv_clean: str, job_clean: str) -> float:
    if not HAS_SKLEARN or not cv_clean or not job_clean:
        return 0.0

    try:
        vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            sublinear_tf=True
        )
        matrix = vectorizer.fit_transform([cv_clean, job_clean])
        similarity = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]
        return float(similarity * 100)

    except Exception as error:
        st.warning(f"Erreur TF-IDF : {error}")
        return 0.0


# ============================================================
# 10. ÉTAPE 9 : JSEARCH API
# ============================================================

def step_9_jsearch_api(job_title: str, api_key: str = "") -> Dict[str, Any]:
    if not HAS_REQUESTS:
        return {
            "status": "UNAVAILABLE",
            "market_demand": "Indisponible",
            "results_found": 0,
            "top_market_skills": []
        }

    if not api_key:
        return {
            "status": "NOT_CONFIGURED",
            "market_demand": "Non configurée",
            "results_found": 0,
            "top_market_skills": []
        }

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "X-RapidAPI-Key": api_key,
        "X-RapidAPI-Host": "jsearch.p.rapidapi.com"
    }
    params = {"query": job_title, "page": "1", "num_pages": "1"}

    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        jobs = data.get("data", [])
        results_count = len(jobs)

        if results_count >= 10:
            demand = "Très Élevée"
        elif results_count >= 5:
            demand = "Élevée"
        elif results_count > 0:
            demand = "Modérée"
        else:
            demand = "Faible"

        return {
            "status": "LIVE_API",
            "market_demand": demand,
            "results_found": results_count,
            "top_market_skills": []
        }

    except Exception as error:
        return {
            "status": "API_ERROR",
            "market_demand": "Indisponible",
            "results_found": 0,
            "top_market_skills": [],
            "error": str(error)
        }


def extract_job_title(job_text: str) -> str:
    lines = [line.strip() for line in job_text.splitlines() if line.strip()]
    return lines[0][:150] if lines else "Data Scientist"


# ============================================================
# 11. ÉTAPE 10 : MATCHING MODEL
# ============================================================

def step_10_matching_model(tfidf_similarity: float, features: Dict[str, float]) -> int:
    tfidf_component = tfidf_similarity * 0.50
    skill_component = (features["skill_match_ratio"] * 100) * 0.40
    soft_component = (features["soft_skills_score"] * 100) * 0.10

    final_score = tfidf_component + skill_component + soft_component
    return int(round(max(0, min(100, final_score))))


def get_score_status(score: int) -> str:
    if score >= 80:
        return "EXCELLENT"
    if score >= 65:
        return "TRÈS BON"
    if score >= 50:
        return "MOYEN"
    return "À AMÉLIORER"


# ============================================================
# 12. SIDEBAR
# ============================================================

with st.sidebar:
    st.markdown("## ⚙️ Paramètres Pipeline")
    st.markdown("### 🌐 JSearch API")

    jsearch_key = st.text_input(
        "Clé RapidAPI JSearch",
        type="password",
        help="Laissez vide pour utiliser le mode simulation."
    )

    st.markdown("---")
    st.markdown("### 🧬 Pipeline AI")

    pipeline_steps = [
        "📄 CV PDF/DOCX", "🔍 Text Extraction", "🧹 Data Cleaning / NLP",
        "🏷️ NER", "🔗 Relationship Extraction", "⚙️ Feature Engineering",
        "📊 TF-IDF", "📐 Cosine Similarity", "🌐 JSearch API",
        "🤖 Matching Model", "🛡️ Score CV AI"
    ]

    for step in pipeline_steps:
        st.caption(step)


# ============================================================
# 13. HEADER
# ============================================================

st.markdown(
    """
    <div class="header-banner">
        <div class="header-title">🛡️ SCORE CV AI</div>
        <div class="header-subtitle">
            Analyse intelligente du CV, extraction NLP, matching avec l'offre,
            comparaison vectorielle et évaluation automatisée du profil candidat.
        </div>
    </div>
    """,
    unsafe_allow_html=True
)


# ============================================================
# 14. VISUALISATION DU PIPELINE
# ============================================================

st.markdown("### 🛣️ Architecture du Pipeline")

c1, c2, c3, c4, c5, c6 = st.columns(6)
p1 = ["📄\nExtraction", "🧹\nNLP", "🏷️\nNER", "🔗\nRelations", "⚙️\nFeatures", "📊\nTF-IDF"]
for col, label in zip([c1, c2, c3, c4, c5, c6], p1):
    with col:
        st.markdown(f'<div class="pipeline-step">{label.replace(chr(10), "<br>")}</div>', unsafe_allow_html=True)

st.markdown("<div style='margin-top: 8px;'></div>", unsafe_allow_html=True)

d1, d2, d3, d4, d5 = st.columns(5)
p2 = ["📐\nCosine", "🌐\nJSearch", "🤖\nMatching", "🛡️\nScore", "📊\nDashboard"]
for col, label in zip([d1, d2, d3, d4, d5], p2):
    with col:
        st.markdown(f'<div class="pipeline-step">{label.replace(chr(10), "<br>")}</div>', unsafe_allow_html=True)

st.markdown("---")


# ============================================================
# 15. FORMULAIRE D'ENTRÉE
# ============================================================

col_cv, col_job = st.columns([1, 1], gap="medium")

DEFAULT_JOB = """
Lead Developer Python, 5 ans d'expérience.
Maîtrise de Docker, Kubernetes et AWS requise.

Compétences clés :
- Python
- Docker
- Kubernetes
- Git
- Data Science
- NLP
- SQL
- Anglais C1

Expérience appréciée avec Spark et le Cloud GCP.
"""

with col_cv:
    with st.container(border=True):
        st.markdown("### 📄 CV Candidat")
        st.caption("Formats acceptés : PDF et DOCX")
        uploaded_file = st.file_uploader("Téléverser le CV", type=["pdf", "docx"], label_visibility="collapsed")

with col_job:
    with st.container(border=True):
        st.markdown("### 💼 Offre d'Emploi")
        job_text = st.text_area("Description du poste", value=DEFAULT_JOB, height=180, label_visibility="collapsed")


# ============================================================
# 16. SCRIPT D'EXÉCUTION DU PIPELINE
# ============================================================

if st.button("🚀 Exécuter le Pipeline SCORE CV AI", use_container_width=True):

    if uploaded_file is None:
        st.warning("⚠️ Veuillez importer un CV.")
        st.stop()

    if not job_text.strip():
        st.warning("⚠️ Veuillez renseigner l'offre.")
        st.stop()

    progress = st.progress(0)
    status = st.empty()

    with st.spinner("Analyse intelligente du CV en cours..."):

        # Étape 1-2
        status.write("📄 Extraction du texte...")
        cv_raw_text = step_1_2_extract_text(uploaded_file)
        progress.progress(10)

        if not cv_raw_text:
            st.error("❌ Aucun texte n'a pu être extrait du CV.")
            st.stop()

        # Étape 3
        status.write("🧹 Nettoyage NLP...")
        cv_clean = step_3_nlp_cleaning(cv_raw_text)
        job_clean = step_3_nlp_cleaning(job_text)
        progress.progress(25)

        # Étape 4
        status.write("🏷️ Extraction des entités (NER)...")
        cv_entities = step_4_ner_extraction(cv_raw_text, cv_clean)
        progress.progress(40)

        # Étape 5
        status.write("🔗 Extraction des relations...")
        relationships = step_5_relationship_extraction(cv_entities, job_clean)
        progress.progress(55)

        # Étape 6
        status.write("⚙️ Feature Engineering...")
        features = step_6_feature_engineering(cv_entities, relationships, cv_clean, job_clean)
        progress.progress(70)

        # Étape 7-8
        status.write("📊 TF-IDF & Cosine Similarity...")
        tfidf_similarity = step_7_8_tfidf_cosine(cv_clean, job_clean)
        progress.progress(80)

        # Étape 9
        status.write("🌐 Analyse du marché JSearch...")
        job_title = extract_job_title(job_text)
        market = step_9_jsearch_api(job_title, jsearch_key)
        progress.progress(90)

        # Étape 10-11
        status.write("🤖 Calculations du Matching Model...")
        final_score = step_10_matching_model(tfidf_similarity, features)

        st.session_state["pipeline_results"] = {
            "score": final_score,
            "tfidf_similarity": tfidf_similarity,
            "entities": cv_entities,
            "relationships": relationships,
            "features": features,
            "market": market
        }

        progress.progress(100)
        status.success("✅ Pipeline SCORE CV AI exécuté avec succès.")


# ============================================================
# 17. DASHBOARD DE RESTITUTION COMPLET
# ============================================================

if "pipeline_results" in st.session_state:

    res = st.session_state["pipeline_results"]

    st.markdown("---")
    st.markdown("## 🛡️ Résultats SCORE CV AI")

    # Ligne 1 : Score & NER Candidat
    col_score, col_candidate = st.columns([1, 1], gap="medium")

    with col_score:
        score = res["score"]
        status_label = get_score_status(score)

        st.markdown(
            f"""
            <div class="score-container">
                <div style="color:#64748b; font-size:11px; font-weight:700; letter-spacing:0.5px;">
                    SCORE GLOBAL DE MATCHING
                </div>
                <div style="display:flex; align-items:baseline; gap:12px;">
                    <div class="score-number">{score}%</div>
                    <span class="score-tag">{status_label}</span>
                </div>
                <div style="color:#475569; font-size:13px; margin-top:6px;">
                    Proximité TF-IDF : <strong>{res['tfidf_similarity']:.1f}%</strong> |
                    Couverture Compétences : <strong>{res['features']['skill_match_ratio']*100:.0f}%</strong>
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with col_candidate:
        with st.container(border=True):
            st.markdown("### 👤 Entités Extrapolées (NER)")
            st.write(f"**Candidat :** {res['entities']['candidate_name']}")
            st.write(f"**Email :** {res['entities']['email']}")
            st.write(f"**Téléphone :** {res['entities']['phone']}")
            st.write(f"**Expérience Détectée :** {res['entities']['years_experience']} ans")

    st.markdown("<br>", unsafe_allow_html=True)

    # Ligne 2 : Matrice des relations & Analyse Marché
    col_rel, col_market = st.columns([1.2, 0.8], gap="medium")

    with col_rel:
        with st.container(border=True):
            st.markdown("### 🔗 Matrice de Compétences (Relationship Extraction)")

            matched = [f'<span class="pill-badge">✓ {r["skill"]}</span>' for r in res["relationships"]["matched_relationships"]]
            missing = [f'<span class="pill-badge-missing">✗ {r["skill"]}</span>' for r in res["relationships"]["missing_relationships"]]

            st.markdown("**Compétences Confirmées :**")
            st.markdown("".join(matched) if matched else "_Aucune compétence directe identifiée._", unsafe_allow_html=True)

            st.markdown("<br>**Écarts / Compétences Manquantes :**", unsafe_allow_html=True)
            st.markdown("".join(missing) if missing else "_Aucun écart majeur détecté._", unsafe_allow_html=True)

    with col_market:
        with st.container(border=True):
            st.markdown("### 🌐 Analyse du Marché (JSearch API)")
            st.write(f"**Statut API :** {res['market']['status']}")
            st.write(f"**Demande du Marché :** {res['market']['market_demand']}")
            st.write(f"**Offres Similaires :** {res['market']['results_found']}")

    # Ligne 3 : Soft Skills & Langues
    st.markdown("<br>", unsafe_allow_html=True)
    col_soft, col_lang = st.columns([1, 1], gap="medium")

    with col_soft:
        with st.container(border=True):
            st.markdown("### 🤝 Soft Skills")
            soft_badges = [f'<span class="pill-badge">{s}</span>' for s in res["entities"]["soft_skills"]]
            st.markdown("".join(soft_badges) if soft_badges else "_Aucune soft skill explicitement détectée._", unsafe_allow_html=True)

    with col_lang:
        with st.container(border=True):
            st.markdown("### 🌍 Langues")
            lang_badges = [f'<span class="pill-badge">🌐 {l}</span>' for l in res["entities"]["languages"]]
            st.markdown("".join(lang_badges) if lang_badges else "_Aucune langue explicitement détectée._", unsafe_allow_html=True)

    # Pied de page
    st.markdown("---")
    st.markdown(
        """
        <div style="text-align: center; color: #64748b; font-size: 12px; padding-bottom: 20px;">
            🛡️ <strong>Score CV AI</strong> • Déployé suivant la chaîne : CV Extraction ➔ NLP ➔ NER ➔ Relationship Extraction ➔ Feature Eng. ➔ TF-IDF ➔ Cosine Similarity ➔ JSearch API ➔ Matching Model
        </div>
        """,
        unsafe_allow_html=True
    )